# March Madness Live Predictor

Batch predict matchups using the O config stacking pipeline.  
Model retrains on all available data when initialized (~10s).

In [2]:
import sys
sys.path.insert(0, '..')
from scripts.live_predict import LivePredictor

In [3]:
# Initialize (loads data + trains, ~10s)
p = LivePredictor(data_dir='../data')

Loading data...
Building features...
Training models (all data through 2025 seasons)...
Ready!



In [24]:
games = {
    'auburn': 'alabama',
    'california baptist': 'southern utah',
    'elon': 'william & mary',
    "mount saint marys": 'siena'
}

In [25]:
p.predict_batch(games)

  No match for 'mount saint marys'
  No match for 'mount saint marys'


,Team1,Team2,Team1_Win%,Team2_Win%
0,Auburn,Alabama,27.1,72.9
1,Cal Baptist,Southern Utah,64.6,35.4
2,Elon,William & Mary,42.5,57.5
3,mount saint marys,siena,NaN,NaN


In [14]:
done = {
    'baylor':'utah',
    'wyoming':'san jose st',
    'fgcu': 'central arkansas',
    'pitt': 'syracuse',
    'kennesaw st': 'new mexico state',
    'california': 'wake forest',
    'boise state': 'colorado st',
    'florida atlantic': 'wichita st',
    'utah state': 'new mexico',
    'njit': 'maine',
    'george mason': 'saint louis',
    'wisconsin': 'purdue',
    'florida': 'kentucky', 
    'georgia': 'mississippi state',
    'drake': 'UIC',
    'jackson state': 'utep',
    'mtsu': 'missouri state',
    'winthrop': 'presbyterian'
}

In [15]:
p.predict_batch(done)

,Team1,Team2,Team1_Win%,Team2_Win%
0,Baylor,Utah,75.9,24.1
1,Wyoming,San Jose St,58.7,41.3
2,FGCU,Cent Arkansas,33.8,66.2
3,Pittsburgh,Syracuse,58.4,41.6
4,Kennesaw,New Mexico St,61.7,38.3
5,California,Wake Forest,48.5,51.5
6,Boise St,Colorado St,60.4,39.6
7,FL Atlantic,Wichita St,53.5,46.5
8,Utah St,New Mexico,41.9,58.1
9,NJIT,Maine,38.2,61.8


## March 7, 2026 — Full Scorecard

All 23 predictions in this notebook were for today's games. Here are results for the 18 completed games from the "done" dict plus CS Fullerton/Northridge.

In [ ]:
import pandas as pd
import numpy as np

# All March 7 actual results matched to notebook predictions
# Prediction is Team1 win probability from our model
results = [
    # "done" dict games
    {'Team1': 'Baylor',       'Team2': 'Utah',          'T1_Pred': 75.9, 'T1_Score': 101, 'T2_Score': 75},
    {'Team1': 'Wyoming',      'Team2': 'San Jose St',   'T1_Pred': 58.7, 'T1_Score': 88,  'T2_Score': 78},
    {'Team1': 'FGCU',         'Team2': 'Cent Arkansas',  'T1_Pred': 33.8, 'T1_Score': 63,  'T2_Score': 73},
    {'Team1': 'Pittsburgh',   'Team2': 'Syracuse',       'T1_Pred': 58.4, 'T1_Score': 71,  'T2_Score': 69},
    {'Team1': 'Kennesaw',     'Team2': 'New Mexico St',  'T1_Pred': 61.7, 'T1_Score': 76,  'T2_Score': 79},
    {'Team1': 'California',   'Team2': 'Wake Forest',    'T1_Pred': 48.5, 'T1_Score': 73,  'T2_Score': 80},
    {'Team1': 'Boise St',     'Team2': 'Colorado St',    'T1_Pred': 60.4, 'T1_Score': 78,  'T2_Score': 67},
    {'Team1': 'FL Atlantic',  'Team2': 'Wichita St',     'T1_Pred': 53.5, 'T1_Score': 70,  'T2_Score': 88},
    {'Team1': 'Utah St',      'Team2': 'New Mexico',     'T1_Pred': 41.9, 'T1_Score': 94,  'T2_Score': 90},
    {'Team1': 'NJIT',         'Team2': 'Maine',          'T1_Pred': 38.2, 'T1_Score': 60,  'T2_Score': 58},
    {'Team1': 'George Mason', 'Team2': 'St Louis',       'T1_Pred': 43.2, 'T1_Score': 86,  'T2_Score': 57},
    {'Team1': 'Wisconsin',    'Team2': 'Purdue',         'T1_Pred': 32.9, 'T1_Score': 97,  'T2_Score': 93},
    {'Team1': 'Florida',      'Team2': 'Kentucky',       'T1_Pred': 75.1, 'T1_Score': 84,  'T2_Score': 77},
    {'Team1': 'Georgia',      'Team2': 'Mississippi St', 'T1_Pred': 69.3, 'T1_Score': 102, 'T2_Score': 96},
    {'Team1': 'Drake',        'Team2': 'UIC',            'T1_Pred': 57.2, 'T1_Score': 51,  'T2_Score': 72},
    {'Team1': 'Jax State',    'Team2': 'UTEP',           'T1_Pred': 35.8, 'T1_Score': 64,  'T2_Score': 61},
    {'Team1': 'MTSU',         'Team2': 'Missouri St',    'T1_Pred': 55.5, 'T1_Score': 75,  'T2_Score': 63},
    {'Team1': 'Winthrop',     'Team2': 'Presbyterian',   'T1_Pred': 68.5, 'T1_Score': 73,  'T2_Score': 71},
    # "games" dict (completed)
    {'Team1': 'CS Fullerton', 'Team2': 'CS Northridge',  'T1_Pred': 42.0, 'T1_Score': 86,  'T2_Score': 79},
]

df = pd.DataFrame(results)
df['T1_Won'] = df['T1_Score'] > df['T2_Score']
df['Pred_T1'] = df['T1_Pred'] / 100
df['Fav'] = df.apply(lambda r: r['Team1'] if r['Pred_T1'] > 0.5 else r['Team2'], axis=1)
df['Winner'] = df.apply(lambda r: r['Team1'] if r['T1_Won'] else r['Team2'], axis=1)
df['Correct'] = ((df['Pred_T1'] > 0.5) & df['T1_Won']) | ((df['Pred_T1'] < 0.5) & ~df['T1_Won'])
df['Brier'] = (df['T1_Won'].astype(float) - df['Pred_T1']) ** 2

# Display
print("=" * 75)
print("MARCH 7 SCORECARD — Model vs Reality")
print("=" * 75)
for _, r in df.iterrows():
    icon = '✅' if r['Correct'] else '❌'
    conf = max(r['Pred_T1'], 1 - r['Pred_T1']) * 100
    print(f"{icon} {r['Team1']:>14s} {r['T1_Score']:3.0f} - {r['T2_Score']:<3.0f} {r['Team2']:<14s} "
          f"| Picked {r['Fav']:<14s} ({conf:.0f}%) | Brier {r['Brier']:.3f}")

print(f"\n{'=' * 75}")
correct = df['Correct'].sum()
total = len(df)
print(f"Record:     {correct}/{total} ({100*correct/total:.1f}%)")
print(f"Mean Brier: {df['Brier'].mean():.4f}")
print(f"Best:       {df.loc[df['Brier'].idxmin(), 'Team1']} vs {df.loc[df['Brier'].idxmin(), 'Team2']} ({df['Brier'].min():.3f})")
print(f"Worst:      {df.loc[df['Brier'].idxmax(), 'Team1']} vs {df.loc[df['Brier'].idxmax(), 'Team2']} ({df['Brier'].max():.3f})")

# Upsets (model wrong, underdog won)
upsets = df[~df['Correct']].copy()
print(f"\nUpsets (model wrong): {len(upsets)}")
for _, r in upsets.iterrows():
    print(f"  {r['Winner']} beat {r['Fav']} — model gave winner only {min(r['Pred_T1'], 1-r['Pred_T1'])*100:.1f}%")

## Predict Historical Games by DayNum

Use `predict_day()` to fetch all games for a given DayNum from our data and auto-score.  
Data goes through DayNum 118 (March 1, 2026). Use `predict_day_range()` for multiple days.

In [ ]:
# Predict all games on DayNum 118 (March 1, 2026) - latest in our data
day_df = p.predict_day(118, season=2026)
print(f"Games on DayNum 118: {len(day_df)}")
print(f"Correct picks: {day_df['Correct'].sum()}/{len(day_df)} ({100*day_df['Correct'].mean():.1f}%)")
print(f"Mean Brier: {day_df['Brier'].mean():.4f}")
day_df

In [ ]:
# Score a range of recent days (last week of data: DayNum 112-118)
range_df = p.predict_day_range(112, 118, season=2026)
range_df